<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [2 · Finding what matches →](02_Finding_What_Matches.ipynb)

# 1 · Rules your data must keep

**The problem.** An address book has rules: a contact is an adult, every phone has a number. Today they live in a
validation function in the Python service, again in the TypeScript form that edits contacts, and again in a sentence in
the interface document. The three drift apart. Someone tightens the age rule in one place, and the others go on
accepting what it now refuses.

mbse-patterns writes such a rule once, as **data**: a *predicate*, kept beside the schemas, stored and sent like any
other data, and applied the same way by every program and language that loads it. This case study writes two rules,
checks data against them, and saves them.

The address book is mbse-schemas' (its tutorial, case study 2): contacts listed in a directory, each with phones. Its
schemas are in `toolkit.py`, beside this notebook.

In [1]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Patterns import Constraints, Predicates, Validators
from mbse.Schemas.Framework import JSON
from toolkit import Contact, Directory, Phone, book, listed

store = book()
ann = store.Contact().name("Ann").age(34).phones(lambda e: e.label("home").phone(lambda p: p.number("555-0100"))).create()
bob = store.Contact().name("Bob").phones(lambda e: e.label("work").phone(lambda p: p)).create()  # no age; a phone, no number
kid = store.Contact().name("Kid").age(9).create()
listed(store, ann, bob, kid)

## Step 1: a predicate

A predicate has a name, **symbols**, each bound to an object of a schema, and a **rule** about them. "A contact is an
adult" has one symbol, `the`, a `Contact`, and the rule `the.age >= 18`. It's built fluently, as a schema is:

In [2]:
IsAnAdult = (
    Predicates.OfPredicate.Builder()
    .name("IsAnAdult")
    .description("18 or older")
    .symbols({"the": Contact})
    .requires(Text.FromFunction(lambda the: the.age >= 18))
    .create()
)
IsAnAdult.name, list(IsAnAdult.symbols), IsAnAdult.rule

('IsAnAdult',
 ['the'],
 _OperationData(name='ge', arguments=(_OperationData(name='get', arguments=(_VariableData(name='the'), _LiteralData(value='age', domain=None)), domain=None), _LiteralData(value=18, domain=None)), domain=None))

`Text.FromFunction` reads the lambda's *source*; it never calls it. The parameter `the` becomes a variable, and the
body becomes an expression of mbse-expressions' Basic dialect: `ge(get(the, 'age'), 18)`. Nothing here is Python any
more, which is what lets TypeScript apply the same rule.

## Step 2: checking data

`Validators.Validate(store, predicates)` checks the predicates on the objects you give it. A predicate applies to a
**match**, a binding of each symbol to an object of its schema; with one symbol, each contact is a match.

In [3]:
validate = Validators.Validate(store, [IsAnAdult])
validate(Contact, ann), validate(Contact, kid), validate(Contact, bob)

([],
 ["the=Contact#0: 'IsAnAdult' does not hold"],
 ["the=Contact#0: 'IsAnAdult' is unknown"])

Ann holds; the kid doesn't. Bob's answer is neither: his age isn't on file, so whether he is 18 or older is
**unknown**, and the validator says so rather than guessing. A plain Python function would have raised, or worse,
compared `None` and failed him. You decide what unknown means for each use:

In [4]:
(Validators.Validate(store, [IsAnAdult], unknown="ignore")(Contact, bob),
 Validators.Validate(store, [IsAnAdult], unknown="violation")(Contact, bob))

([], ["the=Contact#0: 'IsAnAdult' does not hold"])

`Reachable` checks every match among what a root reaches, here the whole directory, and labels each problem by the
object it's about, as mbse-schemas' validator does:

In [5]:
validate.Reachable(Directory, store.singleton("book.Directory"))

["the=Contact#2: 'IsAnAdult' is unknown",
 "the=Contact#3: 'IsAnAdult' does not hold"]

## Step 3: a rule about two objects

"Every phone a contact owns has a number" is about two objects at once: a contact `c` and a phone `p`. With several
symbols, the matches are every **combination** of their objects, so the rule must say how they're related: that `p`
is one of `c`'s phones. `Predicates.Contains(c.phones, ...)` says it, reading the condition on each entry of `c`'s
`phones`:

In [6]:
c, p = E.variable("c"), E.variable("p")
OwnedNumbered = (
    Predicates.OfPredicate.Builder()
    .name("OwnedNumbered")
    .symbols({"c": Contact, "p": Phone})
    .requires(E.operation("implies", Predicates.Contains(c.phones, lambda e: e.phone == p), p.has("number")))
    .create()
)
Validators.Validate(store, [IsAnAdult, OwnedNumbered]).Reachable(Directory, store.singleton("book.Directory"))

["the=Contact#2: 'IsAnAdult' is unknown",
 "the=Contact#3: 'IsAnAdult' does not hold",
 "c=Contact#2, p=Phone#5: 'OwnedNumbered' does not hold"]

Bob's phone has no number. Every other pair of a contact and a phone, Ann and Bob's phone say, satisfies `implies`
trivially, because the phone isn't hers.

## Step 4: rules that can't be right are refused up front

A validator checks its predicates when it's made, so a rule that could never work is reported once, not once per
match:

In [7]:
Broken = Predicates.OfPredicate.Builder().name("Broken").symbols({"c": Contact}).requires(c.age.ge(E.variable("n"))).create()
try:
    Validators.Validate(store, [Broken])
except ValueError as error:
    print(error)

predicate 'Broken': rule: argument 1: variable 'n' is not bound


## Step 5: rules are data

`Constraints.check` gathers predicates in a set. A set saves like any object graph: `JSON.ToJSON` writes it, with each
symbol's schema by name, so any program that has the schemas can read the rules back:

In [8]:
rules = Constraints.check([IsAnAdult, OwnedNumbered])
text = JSON.ToJSON(Constraints.Builders).Reachable(Constraints.OfSet.Schema, rules)
print(text[:400], "...")

{"root": "s0", "objects": {"s0": {"kind": "set", "arguments": [{"argument": {"$ref": "s1", "$schema": "Patterns.Predicate"}, "index": 0}, {"argument": {"$ref": "s2", "$schema": "Patterns.Predicate"}, "index": 1}]}, "s1": {"kind": "predicate", "name": "IsAnAdult", "description": "18 or older", "symbols": [{"name": "the", "type": {"named": {"name": "Contact"}}}], "arguments": [{"argument": {"$ref":  ...


In [9]:
copy = JSON.FromJSON(Constraints.OfStore(store)).Reachable(Constraints.OfSet.Schema, text)
[p.name for p in copy.predicates], copy.predicates[0].symbols["the"] is Contact

(['IsAnAdult', 'OwnedNumbered'], True)

Reading resolves `Contact` by name in the store you give it (`Constraints.OfStore(store)`), so the copy's symbols are
your schemas, not lookalikes. TypeScript reads the same text and gets the same rules.

## Why it works this way

- **Rules beside the schemas, not inside them.** Several sets of rules can apply to one schema: what the editing form
  insists on isn't what the archive tolerates. A schema stays a description of data; a program chooses its rules.
- **Unknown is not false.** Absent isn't `None` in mbse-schemas, and a rule about an absent property has no answer.
  Treating it as false silently fails good records; treating it as true silently passes bad ones.
- **Matches, not loops.** A predicate with two symbols applies to every pair. You don't write the loop; you say how the
  objects are related, and the next case study shows that this is also what makes the rule fast to query.

<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [2 · Finding what matches →](02_Finding_What_Matches.ipynb)